# diagnose_monot5_2023 — does a general MS-MARCO ranker beat our overfit ensemble on 2023?

Reranks the **same cached 2023 NQS pool** with base `castorini/monot5-3b-med-msmarco` (MS-MARCO +
Med-MARCO ranking-pretrained, **zero domain tuning**) and compares to the multi-view ensemble (0.399).

**Hypothesis** (from the reranking-transfer diagnosis): the ensemble collapses on the 2023 questionnaire
format because its rerankers were fit to 2021 narratives and lack a general ranking prior. If a
zero-shot general ranker scores **higher** than 0.399 here, that confirms the missing ingredient is
large-scale ranking pretraining — not domain knowledge. (Clean out-of-domain contrast: our clf_R *beats*
monoT5-MED in-domain on TREC21 judged-pool, ~0.66 vs ~0.45 — so if monoT5-MED wins on 2023, the flip is
the whole point.)

Same pool as the ensemble → any difference is purely the ranker. Requires the `eval_external_2023`
caches on Drive: `trec2023/{pool_nqs_2023.json, doc_fulltext_2023.jsonl, topics2023_text.jsonl, qrels2023.txt}`.
Resumable + cached (`monot5_med_scores_2023.jsonl`).

In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q transformers accelerate sentencepiece pytrec_eval tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json
os.environ['HF_HUB_DISABLE_XET'] = '1'; os.environ['HF_HOME'] = '/content/hf_cache'
import numpy as np, torch
from tqdm.auto import tqdm
from ctmatch.experiments import ExperimentConfig, truncated_doc_text, pytrec_metrics
DATA_ROOT = '/content/drive/MyDrive/ct_data23'; T23 = f'{DATA_ROOT}/trec2023'
cfg = ExperimentConfig(data_root=DATA_ROOT, pool_tag='nqs')   # elig_first-L512, same repr as the eval
BASE = 'castorini/monot5-3b-med-msmarco'
P_SCORES = f'{T23}/monot5_med_scores_2023.jsonl'

id2fields = {}
for l in open(f'{T23}/doc_fulltext_2023.jsonl'):
    r = json.loads(l); id2fields[r.get('nct_id') or r.get('doc_id')] = r
topics = {r['topic_id']: r['topic_text'] for r in map(json.loads, open(f'{T23}/topics2023_text.jsonl'))}
rel = {}
for l in open(f'{T23}/qrels2023.txt'):
    t, _, d, r = l.split(); rel.setdefault(t, {})[d] = int(r)
topics = {t: x for t, x in topics.items() if t in rel}
pool = json.load(open(f'{T23}/pool_nqs_2023.json'))
print('topics', len(topics), '| mean pool/topic', int(np.mean([len(pool[t]) for t in topics])))

In [ ]:
# Base monoT5-MED (general MS-MARCO ranker). Doc rendered elig_first-L512, with 'Relevant:' preserved.
from transformers import T5Tokenizer, T5ForConditionalGeneration
mt_tok = T5Tokenizer.from_pretrained(BASE)
mt = T5ForConditionalGeneration.from_pretrained(BASE, torch_dtype=torch.float16, device_map='auto').eval()
TRUE  = mt_tok('true',  add_special_tokens=False).input_ids[0]
FALSE = mt_tok('false', add_special_tokens=False).input_ids[0]

def render(topic, fields):
    # reserve the scaffold + query so the document truncates (not the 'Relevant:' suffix monoT5 needs)
    reserve = len(mt_tok.encode(f'Query: {topic} Document:  Relevant:', add_special_tokens=False)) + 2
    doc = truncated_doc_text(mt_tok, fields, cfg, reserve=reserve, max_length=512)
    return f'Query: {topic} Document: {doc} Relevant:'

@torch.no_grad()
def score(topic, docs, batch=16):
    out = []
    for i in range(0, len(docs), batch):
        prompts = [render(topic, id2fields[d]) for d in docs[i:i+batch]]
        enc = mt_tok(prompts, return_tensors='pt', padding=True, truncation=True, max_length=512).to(mt.device)
        dec = torch.zeros((enc['input_ids'].shape[0], 1), dtype=torch.long, device=mt.device)
        lg = mt(**enc, decoder_input_ids=dec).logits[:, 0, :]
        lp = torch.log_softmax(lg.float(), -1)
        out.extend((lp[:, TRUE] - lp[:, FALSE]).cpu().tolist())
    return out
print('monoT5-MED loaded')

In [ ]:
# Score the whole 2023 pool (resumable, per-topic).
scores, done = {}, set()
if os.path.exists(P_SCORES):
    for l in open(P_SCORES):
        r = json.loads(l); scores[(r['topic_id'], r['doc_id'])] = r['score']; done.add(r['topic_id'])
todo = [t for t in topics if t not in done]
if todo:
    with open(P_SCORES, 'a') as f:
        for t in tqdm(todo, desc='monoT5-MED score'):
            docs = [d for d in pool[t] if d in id2fields]
            for d, s in zip(docs, score(t, docs)):
                scores[(t, d)] = s; f.write(json.dumps({'topic_id': t, 'doc_id': d, 'score': float(s)}) + '\n')
            f.flush()
print('scored', len(scores), 'pairs')

In [ ]:
# Rerank the pool by monoT5-MED, score, compare.
import pytrec_eval
run = {t: {d: scores[(t, d)] for d in pool[t] if (t, d) in scores} for t in topics}
qrels = {t: {d: int(r) for d, r in rel[t].items()} for t in run}
m = pytrec_metrics(run, qrels, k=10)
per = pytrec_eval.RelevanceEvaluator(qrels, {'ndcg_cut.10'}).evaluate(run)
vals = np.array([per[t]['ndcg_cut_10'] for t in per])
boot = [np.mean(np.random.default_rng(i).choice(vals, len(vals), replace=True)) for i in range(10000)]
print('=== monoT5-MED (zero-shot general ranker) on the 2023 NQS pool ===')
print(m, '| 95% CI', [round(float(np.percentile(boot, 2.5)), 4), round(float(np.percentile(boot, 97.5)), 4)])
print(f'\n  monoT5-MED NDCG@10 = {vals.mean():.4f}')
print(f'  our ensemble       = 0.3988')
print(f'  oracle (pool ceil) = 1.000')
print(f'  IELAB open runs    = 0.576 - 0.672')
d = vals.mean() - 0.3988
if d > 0.02:
    print(f'\n=> CONFIRMED (+{d:.3f}): a zero-shot general ranker beats the overfit ensemble out-of-domain.')
    print('   Missing ingredient = large-scale ranking pretraining, not domain knowledge.')
elif d < -0.02:
    print(f'\n=> NOT confirmed ({d:.3f}): the general ranker is also weak here — the gap is deeper than')
    print('   ranking prior (topic-format / content mismatch), so a general reranker alone will not fix it.')
else:
    print(f'\n=> Inconclusive ({d:+.3f}): roughly tied — see the per-topic split below.')

# per-topic: is monoT5 uniformly better, or spiky? (tells us if it is a robust prior or luck)
ens_hint = 'run eval_external_2023 per-topic to compare directly'
print('\nmonoT5-MED per-topic NDCG@10 (sorted):', sorted([round(v,2) for v in vals]))